<a href="https://colab.research.google.com/github/Daprosero/Domain_Adaptation/blob/main/Images/Notebooks/Tables_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

!git clone https://github.com/ZhangJUJU/OfficeCaltechDomainAdaptation
!pip install -q gdown
!gdown --id 1G0x-arLRPuE-IKDfkxSa-bvPMltvYVsf -O ImageCLEF-DA.zip
!git clone https://github.com/Daprosero/Domain_Adaptation.git


def get_latex_rows(df, model_name):
    rows = []
    for method in methods:
        row_values = []
        method_mask = df['Dataset'].str.contains(method)

        for src, tgt in transfers:
            match = df[method_mask & (df['Source'] == src) & (df['Target'] == tgt)]
            if not match.empty:
                acc = match.iloc[0]['Test Accuracy']
                std = match.iloc[0]['std']
                row_values.append(f"{acc:.2f}~$\\pm$~{std:.2f}")
            else:
                row_values.append("--")

        avg_acc = df[method_mask]['Test Accuracy'].mean()
        avg_std = df[method_mask]['std'].mean()
        row_values.append(f"{avg_acc:.2f}~$\\pm$~{avg_std:.2f}")

        rows.append((model_name, method, row_values))
    return rows

Cloning into 'OfficeCaltechDomainAdaptation'...
remote: Enumerating objects: 2581, done.
remote: Total 2581 (delta 0), reused 0 (delta 0), pack-reused 2581 (from 1)
Receiving objects: 100% (2581/2581), 72.20 MiB | 18.76 MiB/s, done.
Resolving deltas: 100% (9/9), done.
Updating files: 100% (2547/2547), done.
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1G0x-arLRPuE-IKDfkxSa-bvPMltvYVsf
From (redirected): https://drive.google.com/uc?id=1G0x-arLRPuE-IKDfkxSa-bvPMltvYVsf&confirm=t&uuid=8980655c-9b11-444d-9c66-06d2bb90acb9
To: /content/ImageCLEF-DA.zip
100% 225M/225M [00:03<00:00, 66.3MB/s]
Cloning into 'Domain_Adaptation'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (61/61), done

In [2]:
path='/content/Domain_Adaptation/Images/Results/'
Digi18=pd.read_csv(path+'Resnet18/MNIST-USPS-SVHN.csv')
Digi50=pd.read_csv(path+'Resnet50/MNIST-USPS-SVHN.csv')
DigiTr=pd.read_csv(path+'Transformer/MNIST-USPS-SVHN.csv')

In [3]:
methods = ['Baseline', 'DANN', 'ADDA', 'CDAN', 'CREDA']
transfers = [('M', 'U'), ('M', 'S'), ('U', 'M'), ('U', 'S'), ('S', 'M'), ('S', 'U')]

rows_18 = get_latex_rows(Digi18, "ResNet-18")
rows_50 = get_latex_rows(Digi50, "ResNet-50")
rows_vit = get_latex_rows(DigiTr, "ViT-Tiny")

all_rows = rows_18 + rows_50 + rows_vit

latex = []
latex.append("\\begin{table}[H]")
latex.append("\\centering")
latex.append("\\caption{Accuracy (\\%) on Digits for unsupervised domain adaptation using different backbone architectures.}")
latex.append("\\resizebox{0.98\\linewidth}{!}{")
latex.append("\\begin{tabular}{@{}lcccccccc@{}}")
latex.append("\\toprule")
latex.append("\\textbf{Model} & \\textbf{Method} & M $\\rightarrow$ U & M $\\rightarrow$ S & U $\\rightarrow$ M & U $\\rightarrow$ S & S $\\rightarrow$ M & S $\\rightarrow$ U & \\textbf{Avg} \\\\")
latex.append("\\midrule")

# Generación con \multirow
current_model = None
for i, (model, method, values) in enumerate(all_rows):
    if model != current_model:
        latex.append(f"\\multirow{{5}}{{*}}{{{model}}} & {method} & " + " & ".join(values) + " \\\\")
        current_model = model
    else:
        latex.append(f" & {method} & " + " & ".join(values) + " \\\\")

latex.append("\\bottomrule")
latex.append("\\end{tabular}")
latex.append("}")
latex.append("\\label{tab:digits}")
latex.append("\\end{table}")
# Resultado final
latex_code = "\n".join(latex)
print(latex_code)

\begin{table}[H]
\centering
\caption{Accuracy (\%) on Digits for unsupervised domain adaptation using different backbone architectures.}
\resizebox{0.98\linewidth}{!}{
\begin{tabular}{@{}lcccccccc@{}}
\toprule
\textbf{Model} & \textbf{Method} & M $\rightarrow$ U & M $\rightarrow$ S & U $\rightarrow$ M & U $\rightarrow$ S & S $\rightarrow$ M & S $\rightarrow$ U & \textbf{Avg} \\
\midrule
\multirow{5}{*}{ResNet-18} & Baseline & 56.73~$\pm$~20.93 & 22.04~$\pm$~14.77 & 76.64~$\pm$~15.49 & 9.68~$\pm$~10.60 & 69.49~$\pm$~16.82 & 74.69~$\pm$~15.88 & 51.55~$\pm$~15.75 \\
 & DANN & 86.20~$\pm$~10.64 & 19.28~$\pm$~13.61 & 80.84~$\pm$~13.65 & 28.65~$\pm$~16.12 & 72.64~$\pm$~15.72 & 70.66~$\pm$~16.33 & 59.71~$\pm$~14.35 \\
 & ADDA & 7.68~$\pm$~9.13 & 30.99~$\pm$~16.93 & 83.30~$\pm$~12.67 & 28.27~$\pm$~16.22 & 74.32~$\pm$~15.31 & 66.27~$\pm$~15.85 & 54.75~$\pm$~13.49 \\
 & CDAN & 81.99~$\pm$~11.92 & 15.08~$\pm$~12.80 & 25.12~$\pm$~15.75 & 14.19~$\pm$~12.72 & 56.42~$\pm$~17.01 & 66.45~$\pm$~17.51 & 

In [6]:
IRes18=pd.read_csv(path+'Resnet18/ImageCLEF.csv')
IRes50=pd.read_csv(path+'Resnet50/ImageCLEF.csv')
ITrans=pd.read_csv(path+'Transformer/ImageCLEF.csv')

In [7]:
methods = ['Baseline', 'DANN', 'ADDA', 'CDAN', 'CREDA']
transfers = [('I', 'P'), ('I', 'C'), ('P', 'I'), ('P', 'C'), ('C', 'I'), ('C', 'P')]
rows_18 = get_latex_rows(IRes18, "ResNet-18")
rows_50 = get_latex_rows(IRes50, "ResNet-50")
rows_vit = get_latex_rows(ITrans, "ViT-Tiny")
all_rows = rows_18 + rows_50 + rows_vit
latex = []
latex.append("\\begin{table}[H]")
latex.append("\\centering")
latex.append("\\caption{Accuracy (\\%) on ImageCLEF-DA for unsupervised domain adaptation using different backbone architectures.}")
latex.append("\\resizebox{0.98\\linewidth}{!}{")
latex.append("\\begin{tabular}{@{}lcccccccc@{}}")
latex.append("\\toprule")
latex.append("\\textbf{Model} & \\textbf{Method} & I $\\rightarrow$ P & I $\\rightarrow$ C & P $\\rightarrow$ I & P $\\rightarrow$ C & C $\\rightarrow$ I & C $\\rightarrow$ P & \\textbf{Avg} \\\\")
latex.append("\\midrule")
current_model = None
for i, (model, method, values) in enumerate(all_rows):
    if model != current_model:
        latex.append(f"\\multirow{{5}}{{*}}{{{model}}} & {method} & " + " & ".join(values) + " \\\\")
        current_model = model
    else:
        latex.append(f" & {method} & " + " & ".join(values) + " \\\\")
latex.append("\\bottomrule")
latex.append("\\end{tabular}")
latex.append("}")
latex.append("\\label{tab:imageclef}")
latex.append("\\end{table}")
latex_code = "\n".join(latex)
print(latex_code)

\begin{table}[H]
\centering
\caption{Accuracy (\%) on ImageCLEF-DA for unsupervised domain adaptation using different backbone architectures.}
\resizebox{0.98\linewidth}{!}{
\begin{tabular}{@{}lcccccccc@{}}
\toprule
\textbf{Model} & \textbf{Method} & I $\rightarrow$ P & I $\rightarrow$ C & P $\rightarrow$ I & P $\rightarrow$ C & C $\rightarrow$ I & C $\rightarrow$ P & \textbf{Avg} \\
\midrule
\multirow{5}{*}{ResNet-18} & Baseline & 58.00~$\pm$~21.71 & 76.83~$\pm$~21.52 & 68.00~$\pm$~19.30 & 76.50~$\pm$~19.05 & 49.83~$\pm$~24.44 & 38.67~$\pm$~25.60 & 61.31~$\pm$~21.94 \\
 & DANN & 60.00~$\pm$~25.00 & 85.56~$\pm$~13.55 & 66.67~$\pm$~16.43 & 78.89~$\pm$~18.04 & 76.67~$\pm$~17.78 & 57.78~$\pm$~23.74 & 70.93~$\pm$~19.09 \\
 & ADDA & 68.89~$\pm$~20.87 & 77.78~$\pm$~11.10 & 71.11~$\pm$~19.09 & 81.11~$\pm$~16.39 & 74.44~$\pm$~14.56 & 58.89~$\pm$~21.62 & 72.04~$\pm$~17.27 \\
 & CDAN & 56.67~$\pm$~23.31 & 62.22~$\pm$~15.50 & 65.56~$\pm$~16.71 & 58.89~$\pm$~16.28 & 68.89~$\pm$~21.62 & 47.78~$\pm$

In [8]:
ORes18=pd.read_csv(path+'Resnet18/Office-Caltech.csv')
ORes50=pd.read_csv(path+'Resnet50/Office-Caltech.csv')
OTrans=pd.read_csv(path+'Transformer/Office-Caltech.csv')

In [9]:
methods = ['Baseline', 'DANN', 'ADDA', 'CDAN', 'CREDA']
transfers = [('A', 'W'), ('A', 'D'), ('W', 'A'), ('W', 'D'), ('D', 'A'), ('D', 'W')]
rows_18 = get_latex_rows(ORes18, "ResNet-18")
rows_50 = get_latex_rows(ORes50, "ResNet-50")
rows_vit = get_latex_rows(OTrans, "ViT-Tiny")
all_rows = rows_18 + rows_50 + rows_vit
latex = []
latex.append("\\begin{table}[H]")
latex.append("\\centering")
latex.append("\\caption{Accuracy (\\%) on Office-31 for unsupervised domain adaptation using different backbone architectures.}")
latex.append("\\resizebox{0.98\\linewidth}{!}{")
latex.append("\\begin{tabular}{@{}lcccccccc@{}}")
latex.append("\\toprule")
latex.append("\\textbf{Model} & \\textbf{Method} & A $\\rightarrow$ W & A $\\rightarrow$ D & W $\\rightarrow$ A & W $\\rightarrow$ D & D $\\rightarrow$ A & D $\\rightarrow$ W & \\textbf{Avg} \\\\")
latex.append("\\midrule")
current_model = None
for i, (model, method, values) in enumerate(all_rows):
    if model != current_model:
        latex.append(f"\\multirow{{5}}{{*}}{{{model}}} & {method} & " + " & ".join(values) + " \\\\")
        current_model = model
    else:
        latex.append(f" & {method} & " + " & ".join(values) + " \\\\")
latex.append("\\bottomrule")
latex.append("\\end{tabular}")
latex.append("}")
latex.append("\\label{tab:office31}")
latex.append("\\end{table}")
latex_code = "\n".join(latex)
print(latex_code)

\begin{table}[H]
\centering
\caption{Accuracy (\%) on Office-31 for unsupervised domain adaptation using different backbone architectures.}
\resizebox{0.98\linewidth}{!}{
\begin{tabular}{@{}lcccccccc@{}}
\toprule
\textbf{Model} & \textbf{Method} & A $\rightarrow$ W & A $\rightarrow$ D & W $\rightarrow$ A & W $\rightarrow$ D & D $\rightarrow$ A & D $\rightarrow$ W & \textbf{Avg} \\
\midrule
\multirow{5}{*}{ResNet-18} & Baseline & 50.51~$\pm$~29.45 & 55.41~$\pm$~25.37 & 54.91~$\pm$~34.50 & 96.82~$\pm$~7.98 & 46.56~$\pm$~34.31 & 78.98~$\pm$~28.90 & 63.86~$\pm$~26.75 \\
 & DANN & 73.33~$\pm$~15.81 & 87.50~$\pm$~12.50 & 67.36~$\pm$~16.68 & 100.00~$\pm$~0.00 & 51.39~$\pm$~17.09 & 84.44~$\pm$~12.29 & 77.34~$\pm$~12.40 \\
 & ADDA & 62.22~$\pm$~26.71 & 87.50~$\pm$~12.50 & 54.17~$\pm$~19.65 & 100.00~$\pm$~0.00 & 59.72~$\pm$~17.45 & 84.44~$\pm$~9.41 & 74.68~$\pm$~14.29 \\
 & CDAN & 64.44~$\pm$~14.72 & 75.00~$\pm$~12.50 & 50.69~$\pm$~21.64 & 100.00~$\pm$~0.00 & 51.39~$\pm$~16.54 & 88.89~$\pm$~12.2